# PANNs in Colab
Select a GPU runtime and run the cells in order. These first cells download the official CNN14 source and AudioSet checkpoint outside the project repository, then check one short waveform. They do not train or evaluate ESC-50 yet. The checkpoint is stored in Google Drive for later experiments in this notebook.

In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

from google.colab import drive

project_dir = Path('/content/soundview')
panns_source_dir = Path('/content/audioset_tagging_cnn')

if project_dir.exists():
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AndreuLopezz/12607559_SoundView-environmental-sound-recognition.git', str(project_dir)], check=True)
if not panns_source_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/qiuqiangkong/audioset_tagging_cnn.git', str(panns_source_dir)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project_dir / 'requirements-panns.txt')], check=True)
sys.path.insert(0, str(project_dir))
drive.mount('/content/drive')
source_commit = subprocess.run(['git', '-C', str(panns_source_dir), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {importlib.metadata.version("torch")}')
print(f'TorchLibrosa: {importlib.metadata.version("torchlibrosa")}')
print(f'Official PANNs source commit: {source_commit}')

In [ ]:
import urllib.request

checkpoint_path = Path('/content/drive/MyDrive/SoundView/pretrained/Cnn14_mAP=0.431.pth')
checkpoint_url = 'https://zenodo.org/records/3987831/files/Cnn14_mAP=0.431.pth?download=1'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# Keep an incomplete download separate from the checkpoint used for experiments.
if not checkpoint_path.exists():
    temporary_path = checkpoint_path.with_suffix('.part')
    with urllib.request.urlopen(checkpoint_url, timeout=120) as source, temporary_path.open('wb') as target:
        downloaded = 0
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            downloaded += len(chunk)
            if downloaded // (50 * 1024 * 1024) != (downloaded - len(chunk)) // (50 * 1024 * 1024):
                print(f'Downloaded {downloaded / (1024 * 1024):.0f} MB')
    temporary_path.replace(checkpoint_path)

if checkpoint_path.stat().st_size < 100_000_000:
    raise ValueError('The downloaded CNN14 checkpoint is unexpectedly small')
print(f'Checkpoint ready in Drive: {checkpoint_path}')

In [ ]:
import numpy as np
import torch

from soundview.panns import load_panns_model, prepare_panns_audio

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before loading PANNs')
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

# A five-second tone checks the model input and output without touching test data.
time = np.arange(44100 * 5, dtype=np.float32) / 44100
audio = 0.1 * np.sin(2 * np.pi * 440 * time)
waveform = torch.from_numpy(prepare_panns_audio(audio)).unsqueeze(0).to(device)
model = load_panns_model(panns_source_dir, checkpoint_path, device)
with torch.no_grad():
    output = model(waveform)

print(f"Embedding shape: {tuple(output['embedding'].shape)}")
print(f"AudioSet scores shape: {tuple(output['clipwise_output'].shape)}")
assert output['embedding'].shape == (1, 2048)
assert output['clipwise_output'].shape == (1, 527)